# L21: Agent系统与多Agent协作


# L21: Agent系统与多Agent协作

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 解释Single-Agent与Multi-Agent系统的核心差异与适用场景
2. 理解多Agent协作的通信协议、任务分配与冲突解决机制
3. 掌握CrewAI、AutoGen等主流Multi-Agent框架的使用方法
4. 设计能够协同完成复杂任务的多Agent系统
5. 分析Multi-Agent系统中的常见问题：循环、死锁、效率优化

## 2. 核心知识点

### 2.1 Single-Agent vs Multi-Agent

**Single-Agent局限**:
- 能力受限于单一模型
- 无法并行处理多任务
- 处理复杂任务时容易陷入局部解

**Multi-Agent优势**:
- 分布式智能：不同Agent专司其职
- 并行执行：多个Agent同时工作
- 能力组合：集多家之长完成复杂任务
- 容错性：一个Agent失败不影响整体

**Multi-Agent架构类型**:
| 架构 | 特点 | 适用场景 |
|------|------|----------|
| 层级式 | 有明确上下级关系 | 复杂任务的逐步分解 |
| 对等式 | Agent之间平等协作 | 去中心化、灵活性要求高 |
| 竞争式 | Agent之间竞争资源 | 资源优化、方案选择 |
| 混合式 | 结合多种模式 | 复杂真实场景 |

### 2.2 Agent间通信协议

**通信内容**:
- **消息传递**: 文本、状态、指令
- **知识共享**: 共享知识库、向量存储
- **任务交接**: Handoff机制

**Handoff协议**:


In [ ]:
class Agent:
    def __init__(self, name, role, tools):
        self.name = name
        self.role = role
        self.tools = tools
        
    def handoff(self, next_agent, context, reason):
        """
        任务交接：将上下文和原因传递给下一个Agent
        """
        return {
            "from": self.name,
            "to": next_agent.name,
            "context": context,
            "handoff_reason": reason,
            "timestamp": datetime.now().isoformat()
        }

# 使用示例
researcher = Agent("研究员", "负责信息搜集", [search_tool, wikipedia_tool])
analyst = Agent("分析师", "负责数据分析", [python_tool, visualization_tool])
writer = Agent("作家", "负责内容撰写", [writing_tool])

# 研究员完成任务后交给分析师
handoff_msg = researcher.handoff(analyst, 
                                  context={"findings": [...], "summary": "..."},
                                  reason="需要数据分析支持")


### 2.3 CrewAI框架实战


In [ ]:
"""
L21-crewai.py
使用CrewAI构建多Agent协作系统
"""

from crewai import Agent, Task, Crew, Process
from langchain.tools import Tool

# 定义工具
def search_web(query):
    """搜索网络"""
    return f"搜索结果：{query}的相关信息..."

def analyze_data(data):
    """分析数据"""
    return f"数据分析完成，结论：..."

search_tool = Tool(name="搜索", func=search_web, description="搜索网络获取信息")
analyze_tool = Tool(name="分析", func=analyze_data, description="分析给定数据")

# 创建Agent
researcher = Agent(
    role="市场研究员",
    goal="搜集并分析市场相关信息",
    backstory="你是一名资深市场研究员，擅长收集和分析市场数据。",
    tools=[search_tool],
    verbose=True
)

analyst = Agent(
    role="数据分析师",
    goal="提供深入的数据洞察",
    backstory="你是一名专业数据分析师，擅长从数据中提取有价值的洞察。",
    tools=[analyze_tool],
    verbose=True
)

writer = Agent(
    role="报告撰写员",
    goal="撰写专业的市场分析报告",
    backstory="你是一名商业写作专家，擅长撰写清晰、专业的分析报告。",
    verbose=True
)

# 定义任务
research_task = Task(
    description="搜集近三年新能源汽车市场销量数据，分析主要厂商市场份额变化",
    agent=researcher,
    expected_output="市场分析报告草稿"
)

analysis_task = Task(
    description="基于研究员提供的资料，深入分析市场趋势和竞争格局",
    agent=analyst,
    expected_output="数据洞察总结"
)

writing_task = Task(
    description="根据研究和分析结果，撰写完整的市场分析报告",
    agent=writer,
    expected_output="完整报告，包含执行摘要、关键发现和建议"
)

# 创建Crew（Agent团队）
crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, writing_task],
    process=Process.sequential  # 顺序执行，也可以用hierarchical
)

# 启动执行
result = crew.kickoff()
print(f"最终输出: {result}")


### 2.4 AutoGen框架


In [ ]:
"""
L21-autogen.py
使用AutoGen构建多Agent对话系统
"""

from autogen import ConversableAgent, UserProxyAgent, GroupChat, GroupChatManager

# 创建Agent
assistant = ConversableAgent(
    name="Assistant",
    system_message="你是一名专业的AI助手，负责回答问题和提供建议。",
    llm_config={"model": "gpt-3.5-turbo", "temperature": 0.7}
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    code_execution_config={"executor": "local"}
)

# 双Agent对话
user_proxy.initiate_chat(
    assistant,
    message="请帮我分析一下当前AI技术的发展趋势。"
)

# 多Agent群聊
groupchat = GroupChat(
    agents=[assistant, user_proxy],
    messages=[],
    max_round=10
)

manager = GroupChatManager(groupchat=groupchat)

# 启动群聊
user_proxy.initiate_chat(manager, message="我们需要讨论一下明年的技术方向。")


### 2.5 任务分配与冲突解决


In [ ]:
"""
L21-task-allocation.py
多Agent任务分配与冲突解决策略
"""

from enum import Enum

class TaskPriority(Enum):
    HIGH = 1
    MEDIUM = 2
    LOW = 3

class Task:
    def __init__(self, name, estimated_time, required_skills, priority=TaskPriority.MEDIUM):
        self.name = name
        self.estimated_time = estimated_time
        self.required_skills = required_skills
        self.priority = priority
        self.assigned_agent = None
        
class TaskAllocator:
    def __init__(self, agents):
        self.agents = agents
        self.agent_capabilities = {a.name: a.skills for a in agents}
        self.agent_workloads = {a.name: 0 for a in agents}
    
    def allocate_task(self, task):
        """根据技能匹配和工作负载分配任务"""
        # 筛选具备所需技能的Agent
        capable_agents = [
            name for name, skills in self.agent_capabilities.items()
            if any(skill in skills for skill in task.required_skills)
        ]
        
        if not capable_agents:
            return None  # 没有Agent能处理此任务
        
        # 选择工作负载最低的Agent
        selected = min(capable_agents, key=lambda name: self.agent_workloads[name])
        
        # 更新工作负载
        self.agent_workloads[selected] += task.estimated_time
        task.assigned_agent = selected
        
        return selected

class ConflictResolver:
    def __init__(self):
        self.strategies = {
            "priority": self.resolve_by_priority,
            "round_robin": self.resolve_round_robin,
            "consensus": self.resolve_by_consensus
        }
    
    def resolve(self, conflict, strategy="priority"):
        return self.strategies[strategy](conflict)
    
    def resolve_by_priority(self, conflict):
        """按优先级解决：高优先级任务优先"""
        return sorted(conflict.tasks, key=lambda t: t.priority.value)[0]
    
    def resolve_round_robin(self, conflict):
        """轮询解决：每个Agent轮流获得任务"""
        # 实现轮询逻辑
        pass
    
    def resolve_by_consensus(self, conflict):
        """共识解决：多Agent投票决定"""
        # 实现投票逻辑
        votes = {}
        for agent in conflict.agents:
            vote = agent.vote(conflict)
            votes[vote] = votes.get(vote, 0) + 1
        return max(votes, key=votes.get)


## 3. 代码示例


In [ ]:
"""
L21-multi-agent-system.py
完整多Agent系统：自动研究→分析→报告
"""

from crewai import Agent, Task, Crew, Process
from langchain.tools import Tool

# ============ 工具定义 ============
def search_google(query):
    """Google搜索"""
    return f"关于'{query}'的搜索结果..."

def analyze_trends(data):
    """趋势分析"""
    return f"趋势分析结果：{data}"

def write_report(content):
    """撰写报告"""
    return f"报告已生成：{content[:100]}..."

def create_visualization(data):
    """创建可视化"""
    return "可视化图表已创建"

# ============ 创建Agent ============
researcher = Agent(
    role="高级研究员",
    goal="收集全面准确的市场信息",
    backstory="你是一名有10年经验的市场研究员，擅长数据收集和分析。",
    tools=[
        Tool(name="搜索", func=search_google, description="搜索网络获取信息"),
    ]
)

analyst = Agent(
    role="数据分析师",
    goal="提供数据驱动的洞察",
    backstory="你是一名专业数据分析师，擅长从复杂数据中提取洞察。",
    tools=[
        Tool(name="分析", func=analyze_trends, description="分析数据趋势"),
        Tool(name="可视化", func=create_visualization, description="创建数据可视化"),
    ]
)

writer = Agent(
    role="报告撰写专家",
    goal="撰写清晰、专业的报告",
    backstory="你是一名商业写作专家，曾为多家财富500强撰写战略报告。",
    tools=[
        Tool(name="写作", func=write_report, description="撰写报告"),
    ]
)

# ============ 定义任务 ============
task1 = Task(
    description="搜集AI行业最新发展动态，包括技术突破、重要事件、市场变化等",
    agent=researcher,
    expected_output="行业动态总结报告（包含关键事件列表）"
)

task2 = Task(
    description="分析研究员提供的资料，识别关键趋势和洞察",
    agent=analyst,
    expected_output="趋势分析报告（含数据支撑的洞察）"
)

task3 = Task(
    description="基于所有分析结果，撰写完整的行业分析报告",
    agent=writer,
    expected_output="完整报告：执行摘要+关键发现+趋势分析+建议"
)

# ============ 创建Crew ============
crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[task1, task2, task3],
    process=Process.sequential,
    verbose=True
)

# ============ 执行 ============
print("=" * 60)
print("启动多Agent协作：研究 → 分析 → 报告")
print("=" * 60)

result = crew.kickoff()
print("\n最终输出:")
print(result)


## 4. 练习题

1. **架构设计**: 设计一个多Agent系统来处理客户投诉，包括：投诉分类Agent、信息提取Agent、解决方案生成Agent、响应生成Agent。画出架构图并说明Agent之间的通信流程。
2. **CrewAI实战**: 使用CrewAI构建一个"论文阅读助手"系统，包括：搜索论文Agent、分析论文Agent、总结Agent、问答Agent。测试在不同场景下的表现。
3. **冲突解决**: 实现一个冲突解决模块，处理两个Agent同时需要同一个资源的情况。实现至少3种冲突解决策略（优先级、轮询、投票）并比较效果。
4. **性能优化**: 分析多Agent系统的性能瓶颈，提出优化方案并实现：任务缓存、Agent池化、异步通信等。
5. **Multi-Agent评估**: 设计一套评估多Agent系统的指标体系（效率、质量、一致性），并对实际系统进行评估。

## 5. 延伸阅读

- **GitHub**: CrewAI — https://github.com/crewAI/crewAI
- **GitHub**: Microsoft AutoGen — https://github.com/microsoft/autogen
- **GitHub**: LangChain Agents — https://github.com/huggingface/peft
- **论文**: Multi-Agent Reinforcement Learning — 经典MARL论文
- **课程**: Multi-Agent Systems — Coursera上有相关课程

---
